# 2D Single-Component Multi-Phase Lattice Boltzmann (LBM) Simulations 
__Author(s):__ Bernard Chang and Masa Prodanovic

__Last Update:__ Feb. 2026

Copyright © 2026 Digital Porous Media Team. All rights reserved.

---

In this hands-on activity, we will demonstrate the general algorithm used to perform single component multiphase (SCMP) lattice Boltzmann simulations. The goal of this exercise is to implement the fundamental components of the SCMP Shan-Chen model and to demonstrate how simulation parameters influence fluid behavior. This example is performed on a 2-dimensional lattice with 9 discrete velocities (D2Q9).

This notebook assumes basic understanding of lattice Boltzmann methods. For a more comprehensive review of basic concepts, please refer to the [5-2-1_lbm_d2q9_bgk](5_simulation/5-2-1_lbm_d2q9_bgk.ipynb) notebook.

Before we get started, let's import some packages


In [2]:
#@title Import Packages
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
import tifffile

import porespy as ps

# # Porespy for generating domains
# try:
#   import porespy as ps
# except ImportError:
#   !pip install scikit-image==0.23.2 "numpy<2.0.0" --force-reinstall
#   !pip install -q porespy==2.2.2
#   import porespy as ps

from skimage.measure import regionprops,label

import sys
import os
from tqdm import tqdm
from typing import Tuple, List, Iterable, Optional, Any

try:
  os.chdir("./LBM_Workshop")
  # sys.path.append("./LBM_Workshop/")
except:
  !git clone https://github.com/BC-Chang/LBM_Workshop.git
  os.chdir("./LBM_Workshop")

from plotting_utils import plot_profile, plot_quiver, plot_streamlines

# Import a timer
from time import perf_counter_ns, sleep

# Import ipywidgets
import ipywidgets as widgets
from ipywidgets import interact, interact_manual, interactive
from IPython.display import display, clear_output

from functools import wraps

# Define a decorator to extend a class with new methods
def extend(cls, *, override=True):
    def decorator(func):
        if not override and hasattr(cls, func.__name__):
            raise AttributeError(
                f"{cls.__name__}.{func.__name__} already exists"
            )

        @wraps(func)
        def method(*args, **kwargs):
            return func(*args, **kwargs)

        setattr(cls, func.__name__, method)
        return method
    return decorator


Cloning into 'LBM_Workshop'...


## Setting up Single-Component Multiphase (SCMP) Lattice Boltzmann Methods

### Background
As in single-phase lattice Boltzmann methods (LBM), the fundamental variable in **kinetic theory**, on which LBM is based, is the particle distribution function $f$. LBM models the mesoscopic dynamics of fluids by evolving $f$ according to the discretized Boltzmann equation.

Conceptually, LBM tracks the probability of molecules having a certain position and velocity (or momentum) over time, subject to molecular streaming, collisions, and force contributions.

SCMP LBM extends this framework by introducing effective interparticle interactions that enable spontaneous phase separation into coexisting phases (liquid/vapor).

One widely used SCMP formulation is the Shan-Chen pseudopotential model, in which non-ideal fluid behavior arises from density-dependent interaction force rather than explicit interface tracking. More on this later...

First, we will define a class object call ```LBM_SCMP()``` where we will initialize several variables we need to start building our simulator, including the discrete velocity vectors and lattice weights.

In [ ]:
class LBM_SCMP:
    """
    Single-Component Multi-Phase Lattice Boltzmann Method using Shan-Chen Model
    """
    def __init__(self, nx: int, ny: int, omega: float = 1.0, cs2: float = 1.0 / 3.0, G: float = -4.0, rho_0: float = 1.0, G_ads: float = -4.0, psi_solid: float = 0.5, body_force: Iterable[float] = (0.0, 0.0)) -> None:
        """
        Initialize SCMP
        Parameters:
        ---
        nx, ny : int
            Grid dimensions
        omega: float
            Relaxation parameter (1/tau). Default = 1.0 (tau = 1.0)
        cs2: float
            Speed of sound squared. Default = 1/3 for D2Q9
        G: float
            Shan-Chen Interaction strength. Default = -4.0
            Note: Negative values are used for the phase separation.
        rho_0: float
            Reference density for pseudopotential function. Default = 1.0
        G_ads: float
            Fluid-solid interaction strength. Default = -0.4
        psi_solid: float
            Solid pseudopotential function. Default = 0.5
        body_force: Iterable[float, float]
            Force components in x and y directions. Default = (0.0, 0.0)
        """
        self.nx = nx
        self.ny = ny
        self.omega = omega
        self.cs2 = cs2
        self.G = G
        self.G_ads = G_ads
        self.tau = 1.0 / omega
        self.c = np.array([
            [0, 0],
            [1, 0], [0, 1], [-1, 0], [0, -1],
            [1, 1], [-1, 1], [-1, -1], [1, -1]], dtype=np.int16)
        self.weights = np.array(
            [4. / 9.,
             1. / 9., 1. / 9., 1. / 9., 1. / 9.,
             1. / 36., 1. / 36., 1. / 36., 1. / 36.], dtype=np.float64)
        self.opposite = np.array(
            [0,
             3, 4, 1, 2,
             7, 8, 5, 6], dtype=np.uint8)
        # Fields
        self.rho = np.ones((nx, ny), dtype=np.float64)
        self.rho_0 = rho_0
        self.u = np.zeros((2, nx, ny), dtype=np.float64)
        self.f = np.zeros((9, nx, ny), dtype=np.float64)
        self.f_eq = np.zeros((9, nx, ny), dtype=np.float64)
        self.psi = np.empty((nx, ny), dtype=np.float64)
        self.psi_s = psi_solid  # self.rho_0 * (1 - np.exp(-1.))

        self.timestep = 0
        self.solid_mask = np.zeros((nx, ny), dtype=bool)
        self.max_velocity = 0.1  # Limit velocity for stability
        self.rho_min_threshold = 1.e-6  # Minimum density threshold
        self.body_force = body_force  # Force components)
    def initialize_density_field(self, rho_init: np.ndarray, u_init: Optional[np.ndarray] = None) -> None:
        """
        Initialize simulation from arbitrary density field
        Parameters:
        ---
        rho_init: ndarray (nx, ny)
            Initial density field
        u_init: ndarray (2, nx, ny), optional
            Initial velocity field. Defaults to zero everywhere
        Returns:
        ---
        None
        """
        if rho_init.shape != (self.nx, self.ny):
            raise ValueError(f"rho_init has shape {rho_init.shape}, which does not match grid shape ({self.nx}, {self.ny})")
        self.rho = rho_init.copy().astype(np.float64)
        if u_init is None:
            self.u = np.zeros((2, self.nx, self.ny), dtype=np.float64)
        else:
            if u_init.shape != (2, self.nx, self.ny):
                raise ValueError(f"u_init has shape {u_init.shape}, which does not match expected (2, {self.nx}, {self.ny})")
            self.u = u_init.copy().astype(np.float64)
        # Initialize equilibrium distribution
        self.f_eq = self.compute_equilibrium(self.rho, self.u)
        self.f = self.f_eq.copy()

    def set_solids_from_mask(self, solid_mask: np.ndarray) -> None:
        """
        Set solid nodes from a boolean mask
        Parameters:
        ---
        solid_mask: ndarray (nx, ny)
            Boolean array where True indicates solid nodes
        Returns:
        ---
        None
        """
        if solid_mask.shape != (self.nx, self.ny):
            raise ValueError(f"solid_mask has shape {solid_mask.shape}, which does not match grid shape ({self.nx}, {self.ny})")
        self.solid_mask = solid_mask.copy()
        # self.f_eq[:, self.solid_mask] = 0
        self.f[:, self.solid_mask] = 0
        self.u[:, self.solid_mask] = 0



### Obtaining Macroscopic Properties

Recall  that macroscopic properties, such as mass and momentum density can be can be obtained from the weighted sum (moments) of the *discrete-velocity distribution function, $f_i(x, t)$*

The mass density is given by the zeroth moment:
$$\rho = \sum f_i(x, t),$$
and the momentum density by the first moment:
$$\rho u(x,t) = \sum c_i f_i(x, t),$$

where $c_i$ are the discrete lattice velocities defined by the chosen lattice.

These relations are identical to those used in single-phase BGK.

**Note:** In the absence of external or interparticle forces, the macroscopic velocity is obtained directly as

$$u(x,t) = \frac{1}{\rho(x, t)}\sum c_i f_i(x, t).$$

When forces are present, as will be the case for the Shan-Chen multiphase model, the macroscopic velocity must be modified to account for momentum exchange. This correction will be introduced later.

In [5]:
@extend(LBM_SCMP)
def compute_macroscopic(self, f: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute macroscopic density and velocity from distribution function
        Parameters:
        ---
        f: ndarray (9, nx, ny)
            Distribution function
        Returns:
        ---
        rho: ndarray (nx, ny)
            Density field
        u: ndarray (2, nx, ny)
            Velocity field
        """
        rho = np.sum(f, axis=0)
        rho = np.clip(rho, self.rho_min_threshold, None)
        u = np.zeros((2, self.nx, self.ny), dtype=np.float64)
        for i in range(9):
            u[0] += f[i] * self.c[i, 0]
            u[1] += f[i] * self.c[i, 1]
        u /= np.maximum(rho, 1e-9)

        return rho, u

### Equilibrium Calculation

In SCMP, the equilibrium distribution function retains the standard isothermal form from the BGK model. 
Recall that after collision, the distributions $f_i$ relax toward the equilibrium distribution, $f_i^{eq}$ as:

$$ f_i^{eq}(x, t) = w_i \rho \left(1 + \frac{u \cdot c_i}{c_s^2} + \frac{(u \cdot c_i)^2}{2c_s^4} - \frac{u \cdot u}{2c_s^2} \right).$$

Assuming the isothermal lattice Boltzmann equation ($p = c_s^2 \rho$) and operating in lattice units ($\Delta x = \Delta t = 1$), the equilibrium distribution function can be simplified to

$$ f_i^{eq}(x, t) = w_i \rho \left(1 + 3(u \cdot c_i) + \frac{9}{2}(u \cdot c_i)^2 - \frac{3}{2} (u \cdot u) \right).$$

**Note:** Non-ideal effects in SCMP enter through the force term. The equilibrium distribution form itself does not change from BGK.



In [ ]:
@extend(LBM_SCMP)
def compute_equilibrium(self, rho: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Compute equilibrium distribution function
    Parameters:
    ---
    rho: ndarray (nx, ny)
        Density field
    u: ndarray (2, nx, ny)
        Velocity field
    Returns:
    ---
    f_eq: ndarray (9, nx, ny)
        Equilibrium distribution
    """
    f_eq = np.zeros((9, self.nx, self.ny))
    u_sq = u[0]**2 + u[1]**2
    for i in range(9):
        u_dot_c = self.c[i, 0] * u[0] + self.c[i, 1] * u[1]
        f_eq[i] = self.weights[i] * rho * (1 + u_dot_c / self.cs2 + (u_dot_c**2) / (2 * self.cs2**2) - u_sq / (2 * self.cs2))
    f_eq[:, self.solid_mask] = 0.0
    return f_eq

### Collision Step

Recall that LBM advances in two main steps: **collision** and **streaming**. In the standard Shan-Chen SCMP model, the collision operator retains the standard BGK form (in lattice units): 

$$\Omega_i(x, t) = -\frac{f_i(x, t) - f_i^{eq}(x, t)}{\tau}.$$

The post-collision distribution function is therefore:

$$\tilde{f_i}(x, t) = f_i(x, t) + \Omega_i(x, t) = f_i(x, t) - \frac{1}{\tau}[f_i(x, t) - f_i^{eq}(x, t)].$$

This operator describes how the molecular collisions relax $f_i$ toward equilibrium ($f_i^{eq}$) at a rate determined by the characteristic relaxation time $\tau$. For reference, the kinematic viscosity of the fluid is related to the relaxation time as $\nu = c_s^2 (\tau - 1/2)$.

**Note:** We use BGK here for simplicity. Other SCMP variants may employ multiple relaxation time (MRT), two relaxation time (TRT), or other collision models.



In [ ]:
@extend(LBM_SCMP)
def collide(self) -> np.ndarray:
    """
    Perform collision step

    Returns:
    ---
    f_post: ndarray (9, nx, ny)
        Post-collision distribution function
    """
    f_post = self.f - self.omega * (self.f - self.f_eq)
    return f_post

### Streaming & Bounceback

The other main step in simulations is the streaming step, which moves post-collision populations to neighboring nodes along direction $c_i$ as

$$f_i(x + c_i, t) = f_i(x, t).$$

Note that this step is identical to the BGK model.

Periodic boundaries are naturally handled by the ```numpy.roll()``` function in Python. 

#### Bounceback Boundary Condition

In this exercise, we impose no-slip at solid boundary nodes using bounceback. Again, the idea is the same as in the single phase BGK model where populations are reflected at walls so that the tangential velocity vanishes.

<img src='https://www.researchgate.net/profile/Robert-Bialik/publication/258843003/figure/fig1/AS:392599690072074@1470614468708/No-slip-boundary-condition-bounce-back-boundary-condition-for-the-collision-with-the.png' height='300'>

In [ ]:
def stream_and_bounceback(self, f_post: np.ndarray) -> np.ndarray:
    """
    Perform streaming step with bounce-back for solids
    Parameters:
    ---
    f_post: ndarray (9, nx, ny)
        Post-collision distribution function
    Returns:
    ---
    f_next: ndarray (9, nx, ny)
        Distribution function after streaming and bounce-back
    """
    # Initialize next distribution function with zeros.
    f_next = np.zeros_like(f_post)
    solid = self.solid_mask
    fluid = ~solid

    for i in range(9):
        src = np.roll(np.roll(f_post[i], self.c[i, 0], axis=0), self.c[i, 1], axis=1)

        # Indicator function for upstream solid
        upstream_solid = np.roll(np.roll(solid, self.c[i, 0], axis=0), self.c[i, 1], axis=1)

        # For fluid cells:
        # - If upstream is fluid, take src (normal streaming)
        # - If upstream is solid, apply bounce-back: reflect local opposite
        normal_mask = fluid & (~upstream_solid)
        bounce_mask = fluid & upstream_solid

        f_next[i][normal_mask] = src[normal_mask]
        f_next[i][bounce_mask] = f_post[self.opposite[i]][bounce_mask]

    # Zero out solids for posterity
    f_next[:, self.solid_mask] = 0  # Zero out solid cells
    return f_next

### Forcing
#### Pseudopotential, $\psi(\rho)$, and Non-Ideal Pressure

The Shan–Chen single-component multiphase model represents non-ideal fluid behavior through a density-dependent "pseudopotential" that induces attraction between neighboring lattice sites. This mechanism generates diffuse interfaces and liquid–vapor phase coexistence without explicit interface tracking.

The core idea is that the local density, $\rho$, maps to a **pseudopotential** field, $\psi(\rho)$. Neighboring sites interact through $\psi$, creating a net attraction or repulsion controlled by an interaction parameter, $G$. This attraction modifies the pressure away from the ideal gas relation, enabling vapor/liquid phase separation and coexistence.

With pseudopotential interactions, the effective equation of state is:

$$ p(\rho) = c_s^2 \rho + \frac{G c_s^2}{2}\psi(\rho)^2.$$

where the ideal contribution ($c_s^2 \rho$) is complemented by an "interaction pressure" term, ($\frac{G c_s^2}{2}\psi(\rho)^2$).

When $G < 0$ and $\psi(\rho)$ is sufficiently increasing, the non-ideal pressure develops a region where $dp/d\rho$ is small or negative. This mechanical instability drives phase separation: locally-denser regions experience stronger attraction forces and draw in more mass. The system evolves toward two coexisting phases - a vapor with density $\rho_g$ and a liquid with density $\rho_l$ that share a common equilibrium pressure $p_0$.

Since the SCMP interface is diffuse, the density varies smoothly from $\rho_g$ to $\rho_l$ over several lattice nodes. Within this interfacial region, neighboring lattice sites experience unequal pseudopotential interactions because one side of the interface contains denser fluid than the other. An effective interfacial tension emerges from these anisotropic contributions. At the macroscopic level, this manifests as surface tension that resists interface deformation and curvature. Importantly, the interfacial tension is not imposed explicitly in the Shan-Chen model; it emerges naturally from the pseudopotential.

One can design pseudopotential functions to recover a desired non-ideal equation of state by matching the interaction pressure term to a prescribed pressure-density relation. A variety of pseudopotential functions have been proposed, with the most common choice (and the one used here) being the original Shan-Chen form,

$$\psi(\rho) = \rho_0 (1 - \exp{(-\rho / \rho_0)}),$$

where $\rho_0$ is commonly taken to be 1.


##### Numerical Considerations

Despite its simplicity and robustness, the Shan–Chen pseudopotential model has well-known numerical limitations.

A common issue is the presence of spurious currents, which appear as small, nonphysical velocity fields near interfaces, especially around curved surfaces. These currents are influenced by lattice discretization errors and tend to grow with increasing interaction strength $|G|$, surface tension, and density ratio. $|G|$ also affects the interfacial thickness, where larger $|G|$ leads to sharper interfaces but exacerbates spurious currents and can ultimately lead to unstable behavior.

Additionally, the standard single-component model typically supports moderate density ratios up to O(10 -100). Other LBM formulations have been designed to address some of these limitations.


In [ ]:
@extend(LBM_SCMP)
def compute_pseudopotential(self, rho: np.ndarray) -> None:
    """
    Compute pseudopotential function psi(rho)
    Parameters:
    ---
    rho: ndarray (nx, ny)
        Density field
    Returns:
    ---
    psi: ndarray (nx, ny)
        Pseudopotential field
    """
    psi = self.rho_0 * (1 - np.exp(-rho / self.rho_0))
    psi[self.solid_mask] = 0.0
    return psi

#### Shan-Chen Forcing

##### Fluid-Fluid Interaction
In the pseudopotential framework, non-ideal fluid behavior is introduced through an additional interaction force that acts locally between neighboring lattice nodes. As discussed above, this pseudopotential is responsible for phase separation and interfacial tension, and is incorporated into the LBM simulation through a forcing scheme.

For a single-component fluid, the Shan-Chen fluid-fluid interaction force is given by:

$$ F_{ff} = -G \psi(x) \sum_i w_i \psi(\rho(x + c_i)) c_i, $$

where $G$ is the interaction strength, $\psi(\rho)$ is the pseudopotential, $\rho(x)$ is the density at position $x$, and $c_i$ and $w_i$ are the lattice velocity directions and weights. The summation is over all neighbors of the current node. The intuition here is that in homogeneous regions, the force contributions cancel by symmetry. In contrast, near density gradients (e.g., at interfacial regions), the force becomes unbalanced and drives phase separation

##### Fluid-Solid Interaction and Wetting
The same interaction concept can be extended to model fluid-solid interactions, enabling the simulation of wetting effects without explicitly imposing boundary conditions on the solid surfaces. There are a few ways to implement this. Here, we treat solid nodes using an indicator function $S(x)$, which takes the value 1 if a node is solid and 0 otherwise. The fluid-solid interaction force can be independently computed and can be added to the fluid-fluid interaction force as:

$$ F_{fs} = -G_{ads} \psi(x) \sum_i w_i \psi_s S(x+c_i) c_i, $$

where $G_{ads}$ is the adhesive interaction strength, and $\psi_s(x)$ is the pseudopotential of the solid phase. By tuning $G_{ads}$ and $\psi_s$, the solid wall can be made effectively attracting (wetting) or repulsive (non-wetting). ​

In [ ]:
@extend(LBM_SCMP)
def compute_shan_chen_force(self) -> np.ndarray:
    """
    Compute Shan-Chen intermolecular force. This force drives phase separation when G < 0.
    Returns:
    ---
    F : ndarray (2, nx, ny)
        Force field [Fx, Fy]
    """
    F = np.zeros((2, self.nx, self.ny), dtype=np.float64)
    psi = self.compute_pseudopotential(self.rho)
    has_wall = (self.solid_mask is not None) and self.solid_mask.any()

    # if has_wall:
    #     psi = np.where(self.solid_mask, 0.0, psi)
    for i in range(1, 9):  # skip rest direction
        ex, ey = int(self.c[i, 0]), int(self.c[i, 1])

        # Neighbor pseudopotential by rolling psi (fluid neighbors)
        psi_nb = np.roll(np.roll(psi, -ex, axis=0), -ey, axis=1)

        w_i = self.weights[i]

        if not has_wall:
            # Pure fluid–fluid SC
            F[0] += -self.G * w_i * ex * psi * psi_nb
            F[1] += -self.G * w_i * ey * psi * psi_nb
        else:
            # Identify whether the neighbor is a solid (rolled solid mask)
            solid_nb = np.roll(np.roll(self.solid_mask, -ex, axis=0), -ey, axis=1)
            fluid_nb = ~solid_nb

            # Fluid–fluid contribution (only where neighbor is fluid)
            F[0] += -self.G * w_i * ex * psi * (psi_nb * fluid_nb)
            F[1] += -self.G * w_i * ey * psi * (psi_nb * fluid_nb)

            # Fluid–solid contribution (use constant psi_wall for solid neighbors)
            # Tip: set self.psi_wall (e.g., 0.2–0.8) and self.G_ads for wetting strength
            F[0] += -self.G_ads * w_i * ex * psi * (self.psi_s * solid_nb)
            F[1] += -self.G_ads * w_i * ey * psi * (self.psi_s * solid_nb)

    # No force inside solids
    if (self.solid_mask is not None):
        F[:, self.solid_mask] = 0.0

    return F

#### Body Forces

In addition to the Shan-Chen interaction force, lattice Boltzmann simulations often include external body forces, such as gravity or pressure-driven force. Body forces act on every fluid node and can be added directly to the total force acting on the fluid (e.g., from Shan-Chen force). 

In practice, a prescribed body force, $b = [b_x, b_y]$, contributes to momentum change at each lattice node. The physical force on a fluid element is then proportional to its mass, which in lattice units is $\rho$. Therefore, the macroscopic body force acting on a fluid node is computed as:

$$ F_{body} = \rho b.$$

Solid nodes are typically excluded, so no body force is applied inside solid regions.

In [7]:
@extend(LBM_SCMP)
def compute_body_force(self) -> np.ndarray:
    """
    Apply body force
    Parameters:
    ---
    Returns:
    ---
    F : ndarray (2, nx, ny)
    """
    b = np.asarray(self.body_force, dtype=np.float64)
    assert b.ndim == 1, f"Body force should be a 1D array of size (2,), got {b.ndim}D"
    F = self.rho[None, :, :] * b[:, None, None]

    if (self.solid_mask is not None) and self.solid_mask.any():
        F[:, self.solid_mask] = 0.0

    return F

#### Guo Forcing

The total force at a node is computed by summing all contributions, e.g.,:

$$ F_{total} = F_{ff} + F_{fs} + F_{body}. $$

The force must be incorporated into the evolution of the distribution functions. One naive approach is to directly add the force contributions to the velocity after streaming; however, this can introduce errors or reduce accuracy.

Guo's forcing sheme provides a more accurate way to include forces in LBM. For a lattice node with total force $F_{total}$, the forcing contribution to the distribution function along the lattice direction $c_i$ is:

$$ F_i = \left(1 - \frac{1}{2\tau}\right)w_i \left[\frac{c_i - u}{c_s^2} + \frac{(c_i \cdot u)c_i}{c_s^4}\right] F_{total}. $$

**IMPORTANT:** The macroscopic velocity, $u$, must include a *half-force correction*:

$$ u = \frac{1}{\rho} \sum_i f_i c_i + \frac{\Delta t}{2\rho} F_{total}. $$

The first term is the usual velocity moment from the particle distributions. The second term accounts for the momentum that will be added by the force during the time step. Here, we perform the half-force velocity correction during the time step function, not in the following Guo forcing computation function.

The Guo forcing term for each lattice direction is then added to the collision step:

$$ f_i(x, t + \Delta t) = f_i(x, t) + \frac{1}{\tau} \left[f_i(x, t) - f_i^{eq}(\rho, u)\right] + F_i. $$

In [8]:
@extend(LBM_SCMP)
def compute_guo_forcing(self, F) -> np.ndarray:
    """
    Compute Guo forcing
    Parameters:
    ---
    F : ndarray (2, nx, ny)
        Shan-Chen force + Body force
    Returns:
    ---
    guo_force : ndarray (2, nx, ny)
        Guo forcing term
    """

    guo_force = np.zeros_like(self.f, dtype=np.float64)
    ci = self.c[:, :, None, None]
    ui = self.u[None, :, :, :]
    Fi_vec = F[None, :, :, :]
    ci_dot_u = np.sum(ci * ui, axis=1)
    term2 = (ci - ui) / self.cs2 + (ci_dot_u[:, None, :, :] * ci) / (self.cs2 ** 2)
    guo_force = self.weights[:, None, None] * (1 - 0.5 * self.omega) * np.sum(term2 * Fi_vec, axis=1)
    return guo_force

### Putting it all together

We now have the necessary functions to run SCMP simulations! In the following cell, we will define a class method to complete one LBM time step. The general workflow for each iteration is:

1. Compute macroscopic variables (density and velocity)
2. Compute the pseudopotential for Shan-Chen interactions
3. Compute all force contributions (Shan-Chen fluid-fluid, Shan-Chen solid-fluid, and body forces)
4. Compute half-force velocity correction
5. Compute equilibrium distributions
6. Compute Guo forcing and apply to the collision step
7. Apply streaming and bounce-back


In [ ]:
@extend(LBM_SCMP)
def step(self):
    """
    Perform one simulation step

    Returns:
    ---
    None
    """
    # Macroscopic variables
    self.rho, u_temp = self.compute_macroscopic(self.f)

    # Compute pseudopotential
    self.psi = self.compute_pseudopotential(self.rho)
    # Compute Shan-Chen force
    F = self.compute_shan_chen_force()

    # Apply body force, if specified
    if self.body_force != (0.0, 0.0):
        F += self.compute_body_force()

    # Update velocity with half-force correction
    self.u = u_temp + 0.5 * F / np.maximum(self.rho, 1e-1)
    if (self.solid_mask is not None) and self.solid_mask.any():
        self.u[:, self.solid_mask] = 0.0

    # Compute equilibrium distribution
    self.f_eq = self.compute_equilibrium(self.rho, self.u)

    # Guo Forcing
    guo_force = self.compute_guo_forcing(F)

    # Collide
    f_new = self.collide() + guo_force

    # Stream and Bounceback
    self.f = self.stream_and_bounceback(f_new)

    self.timestep += 1

###  Read in the geometry
Before we get into the flow simulation, let's first read in our geometry. For this exercise, we assume a 2D binary image with:
- 0 indicating fluid space
- 1 indicating solid space

For this workshop, we select an image from the data folder. You can also load in your own image or create one yourself if you'd like.


In [ ]:
Nx = 150
Ny = 150
X, Y = np.meshgrid(range(Nx), range(Ny))
data = (X - Nx/4)**2 + (Y - Ny/2)**2 < (Ny/8)**2

plt.imshow(data, cmap='binary')
plt.colorbar()


Let's run our LBM simulation! This simulation does not fully converge in the default number of iterations, but it gets the point across. This should take about one minute to run.

In [ ]:
u_x, u_y, u = run_lbm(data)

In [ ]:
_ = plot_profile(u, cmap='jet')

In [ ]:
#@title Read in data
geom_options = os.listdir("./data/")

# TODO create catalog of images, read in from drop down, and plot.
data_dropdown = widgets.Dropdown(
    concise=False,
    options=geom_options,
    value='beads.tif',
    description='Select a file to read in'
)

# select_button = widgets.Button(description='I want this one!')
# output = widgets.Output()
def read_in_and_plot(dropdown):
  display(dropdown)
  data = tifffile.imread(os.path.join("./data/", dropdown))
  clear_output(wait=True)
  display(dropdown)
  plt.imshow(data, cmap='binary')
  plt.colorbar()
  plt.show()

  return data

widget = widgets.interactive(read_in_and_plot, dropdown=geom_options)
display(widget)


In [ ]:
data = widget.result
u_x, u_y, u = run_lbm(data)


In [ ]:
# from plotting_utils import plot_quiver
profile_fig = plot_profile(u, cmap='jet')

## Calculate Permeability

Once we have the velocity field, we can compute the absolute permeability using Darcy's law:

$$k = \frac{\bar{u} \mu L}{\Delta P}$$


Keep in mind that the calculated permeability will be in lattice units ($lu^2$) To convert to the true permeability, we would need to know the physical size of our grid sizes.

For sake of simplicity, we can calculate the mean velocity of the flow-direction-component.

In [ ]:
def vel_avg(ux):
  u_mean = torch.mean(ux[ux != 0])
  print(f"Average Velocity = {u_mean}")
  return

# Calculate the average velocity for our image
vel_avg(u_x)

Let's use the Porespy library to generate some blobs. Make sure you get a geometry that percolates!

In [ ]:
def generate_blobs(phi):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  obstacle = torch.tensor(~ps.generators.blobs(shape=[200, 200], porosity=phi)).to(device)
  #obstacle = torch.tensor(fromfunction(obstacle_fun(cx,cy,r),(nx, ny)))
  plt.imshow(obstacle.cpu(), cmap='binary')
  plt.colorbar()
  return obstacle

phi_widget = widgets.FloatSlider(value=0.5, min=0.5, max=0.86, description='Porosity',
                                 continuous_update=False, step=0.01)

blobs = widgets.interactive(generate_blobs, {'manual': True},
                                       phi=phi_widget)

blobs


In [ ]:
u_x, u_y, u = run_lbm(blobs.result)

In [ ]:
profile_fig = plot_profile(u_x, cmap='jet')

In [ ]:
# Average velocity
vel_avg(u_x)

# Multiphase Lattice Boltzmann Method

Today, we will use the Shan-Chen model, where multiple lattices are superimposed and particle distributions interact via pseudo-potential interparticle forces controlled by $G_c$. Selection of $G_c$ induces interfacial tension and can determine wettability.

Other models include:
* Color Model
* Free Energy Based models


The Shan-Chen model is implmented in our in-house code — [MPLBM-UT](https://doi.org/10.1016/j.softx.2022.101097).

The MPLBM Workflow

<img src='https://ars.els-cdn.com/content/image/1-s2.0-S2352711022000668-gr1.jpg'>


### Let's move on to MPLBM-UT on TACC!

